# Assignment 1: Python Fundamentals for NLP
**Building a Text Statistics Analyzer using Python Fundamentals**

This notebook uses the BBC dataset files:
- `bbc.docs`
- `bbc.terms`
- `bbc.mtx`
- `bbc.classes`

It covers:
- File loading and exploration
- Regex pattern extraction
- NumPy statistics
- Pandas analysis
- Matplotlib visualizations
- Zipf's Law (Bonus)


In [ ]:

import re
import json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from collections import Counter
from scipy.io import mmread
from scipy.stats import norm

DOCS_FILE = "bbc.docs"
TERMS_FILE = "bbc.terms"
MTX_FILE = "bbc.mtx"
CLASSES_FILE = "bbc.classes"

def load_terms(path):
    with open(path, "r", encoding="utf-8", errors="ignore") as f:
        return [x.strip() for x in f.readlines()]

def load_docs(path):
    with open(path, "r", encoding="utf-8", errors="ignore") as f:
        return [x.strip() for x in f.readlines()]

terms = load_terms(TERMS_FILE)
docs = load_docs(DOCS_FILE)

matrix = mmread(MTX_FILE).tocsr()

print("Documents:", len(docs))
print("Terms:", len(terms))
print("Matrix shape:", matrix.shape)


In [ ]:

# Create pseudo text documents from term-frequency matrix

documents = []

for i in range(matrix.shape[0]):
    row = matrix.getrow(i)
    words = []
    for idx, freq in zip(row.indices, row.data):
        words.extend([terms[idx]] * int(freq))
    documents.append(" ".join(words))

print("Sample document length:", len(documents[0].split()))


In [ ]:

# Part 1 - Basic Statistics

stats = []

for i, text in enumerate(documents):
    words = re.findall(r'\b\w+\b', text)
    sentences = re.split(r'[.!?]+', text)

    stats.append({
        "document_id": i+1,
        "filename": docs[i],
        "char_count": len(text),
        "word_count": len(words),
        "sentence_count": len([s for s in sentences if s.strip()]),
        "avg_word_length": np.mean([len(w) for w in words]) if words else 0
    })

df_stats = pd.DataFrame(stats)
df_stats.head()


In [ ]:

df_stats.to_csv("document_statistics.csv", index=False)
print("CSV exported")


In [ ]:

# Part 2 - Regex Extraction

sample_text = "Contact test@example.com or admin@bbc.co.uk on +1-123-456-7890. Visit https://bbc.com on 01/01/2025 and pay $100."

emails = re.findall(r'\b[A-Za-z0-9._%+-]+@[A-Za-z0-9.-]+\.[A-Za-z]{2,}\b', sample_text)

phones = re.findall(r'(?:\+1-)?\d{3}-\d{3}-\d{4}|\(\d{3}\)\s*\d{3}-\d{4}', sample_text)

urls = re.findall(r'https?://[^\s]+', sample_text)

dates = re.findall(r'\d{2}/\d{2}/\d{4}', sample_text)

currency = re.findall(r'[$£€]\d+(?:\.\d+)?', sample_text)

patterns = {
    "emails": emails,
    "phones": phones,
    "urls": urls,
    "dates": dates,
    "currency": currency
}

with open("extracted_patterns.json","w") as f:
    json.dump(patterns,f,indent=4)

patterns


In [ ]:

# Part 3 - Word Frequency Statistics

all_words = []

for d in documents:
    all_words.extend(re.findall(r'\b\w+\b', d.lower()))

freq = Counter(all_words)

freq_values = np.array(list(freq.values()))

print("Mean:", np.mean(freq_values))
print("Median:", np.median(freq_values))
print("Std:", np.std(freq_values))
print("25th:", np.percentile(freq_values,25))
print("50th:", np.percentile(freq_values,50))
print("75th:", np.percentile(freq_values,75))


In [ ]:

# Character Analysis

chars = Counter("".join(all_words))

vowels = sum(chars.get(c,0) for c in "aeiou")
consonants = sum(v for k,v in chars.items() if k.isalpha() and k not in "aeiou")

print("Most common:", chars.most_common(10))
print("Vowel/Consonant Ratio:", vowels/consonants)


In [ ]:

# Visualization 1 - Top 20 Words

top20 = freq.most_common(20)

words = [x[0] for x in top20]
counts = [x[1] for x in top20]

plt.figure(figsize=(10,6))
plt.barh(words, counts)

for i,v in enumerate(counts):
    plt.text(v, i, str(v))

plt.title("Top 20 Words")
plt.show()


In [ ]:

# Visualization 2 - Word Length Histogram

lengths = [len(w) for w in all_words]

plt.figure(figsize=(8,5))
plt.hist(lengths, bins=20, density=True)

mu = np.mean(lengths)
sigma = np.std(lengths)

x = np.linspace(min(lengths), max(lengths), 100)
plt.plot(x, norm.pdf(x, mu, sigma))

plt.title("Word Length Distribution")
plt.show()


In [ ]:

# Visualization 3 - Character Frequency

top_chars = Counter(dict(chars.most_common(15)))

plt.figure(figsize=(10,5))
plt.bar(top_chars.keys(), top_chars.values())
plt.title("Top Characters")
plt.show()


In [ ]:

# Visualization 4 - Sentence Length Box Plot

sentence_lengths = [len(doc.split()) for doc in documents]

plt.figure(figsize=(6,4))
plt.boxplot(sentence_lengths)
plt.title("Sentence Length Distribution")
plt.show()


In [ ]:

# Visualization 5 - Dashboard

fig, ax = plt.subplots(2,2, figsize=(12,10))

ax[0,0].bar(words, counts)
ax[0,0].tick_params(axis='x', rotation=90)

ax[0,1].hist(lengths, bins=20)

ax[1,0].bar(list(top_chars.keys()), list(top_chars.values()))

ax[1,1].boxplot(sentence_lengths)

plt.tight_layout()
plt.show()


In [ ]:

# Part 5 - Pandas Analysis

word_df = pd.DataFrame({
    "word": list(freq.keys()),
    "frequency": list(freq.values())
})

word_df["length"] = word_df["word"].str.len()
word_df["first_char"] = word_df["word"].str[0]
word_df["vowel_count"] = word_df["word"].str.count(r'[aeiou]')

word_df.head()


In [ ]:

group_length = word_df.groupby("length")["frequency"].agg(["mean","max","count"])
group_first = word_df.groupby("first_char")["frequency"].sum()

top50 = word_df.sort_values("frequency", ascending=False).head(50)
long_words = word_df[word_df["length"] > 10]

pivot = pd.pivot_table(word_df,
                       values="frequency",
                       index="length",
                       columns="first_char",
                       aggfunc="sum",
                       fill_value=0)

print(top50.head())
print(long_words.head())
pivot.head()


In [ ]:

# Bonus - Zipf Law

rank_freq = sorted(freq.values(), reverse=True)

plt.figure(figsize=(8,5))
plt.loglog(range(1, len(rank_freq)+1), rank_freq)
plt.title("Zipf's Law")
plt.xlabel("Rank")
plt.ylabel("Frequency")
plt.show()
